# Ingestão de dados via API RIPE Atlas

**Objetivo:** consumir resultados de uma medição pública da API RIPE Atlas, receber os dados em JSON, transformá-los em um `DataFrame` e salvar a coleta no Google Drive.

> Este notebook cobre **somente a ingestão e o armazenamento da camada `raw`**. Não agregue janelas, não crie variáveis, não rotule dados e não treine modelos nesta etapa.

## Entrega mínima

- Justificar as decisões da equipe sobre os parâmetros da consulta;
- Executar uma requisição válida à API;
- Exibir os dados em um `DataFrame`;
- Salvar o JSON original da resposta;
- Salvar os registros em **CSV ou Parquet** na pasta `raw` do Google Drive;
- Registrar evidências da coleta: URL, parâmetros, horário, quantidade de registros e nomes dos arquivos.


## Pipeline da fase de coleta e ingestão

O desenho abaixo apresenta o percurso desta etapa, desde a escolha da medição pública até a verificação dos arquivos salvos na pasta `raw`. Se a consulta não for válida, a equipe deverá revisar os parâmetros e realizar uma nova tentativa.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon

# Desenho do pipeline da fase de coleta e ingestão
fig, ax = plt.subplots(figsize=(11, 16))
ax.set_xlim(0, 11)
ax.set_ylim(0, 18)
ax.axis("off")

azul = "#DCEEFF"
verde = "#DDF5E5"
laranja = "#FFF0D9"
vermelho = "#FFE1E1"
borda = "#315A7D"

def caixa(x, y, texto, cor=azul, largura=5.8, altura=0.8):
    forma = FancyBboxPatch(
        (x - largura / 2, y - altura / 2), largura, altura,
        boxstyle="round,pad=0.10,rounding_size=0.12",
        linewidth=1.7, edgecolor=borda, facecolor=cor
    )
    ax.add_patch(forma)
    ax.text(x, y, texto, ha="center", va="center", fontsize=11, weight="bold")

def decisao(x, y, texto):
    largura, altura = 3.4, 1.05
    pontos = [(x, y + altura/2), (x + largura/2, y),
              (x, y - altura/2), (x - largura/2, y)]
    ax.add_patch(Polygon(pontos, closed=True, edgecolor=borda,
                         facecolor=laranja, linewidth=1.7))
    ax.text(x, y, texto, ha="center", va="center", fontsize=10.5, weight="bold")

def seta(x1, y1, x2, y2, rotulo=None, deslocamento=(0, 0)):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", lw=1.8, color=borda))
    if rotulo:
        ax.text((x1+x2)/2 + deslocamento[0], (y1+y2)/2 + deslocamento[1],
                rotulo, fontsize=10, weight="bold", color="#334155",
                ha="center", va="center")

ax.text(5.5, 17.5, "Pipeline da fase de coleta e ingestão",
        ha="center", fontsize=17, weight="bold", color="#173B57")
ax.text(5.5, 17.05, "API RIPE Atlas → camada raw no Google Drive",
        ha="center", fontsize=11, color="#475569")

etapas = [
    (16.2, "1. Pesquisar medições públicas", azul),
    (14.9, "2. Escolher o measurement_id", azul),
    (13.6, "3. Definir o intervalo da coleta", azul),
    (12.3, "4. Registrar e justificar os parâmetros", azul),
    (11.0, "5. Enviar requisição à API RIPE Atlas", verde),
]
for y, texto, cor in etapas:
    caixa(5, y, texto, cor)

for y1, y2 in [(15.8, 15.3), (14.5, 14.0), (13.2, 12.7), (11.9, 11.4)]:
    seta(5, y1, 5, y2)

decisao(5, 9.65, "6. Resposta válida?")
seta(5, 10.6, 5, 10.2)

# Caminho de erro e nova tentativa
caixa(9.0, 8.3, "Revisar ID, período\nou conexão", vermelho, largura=2.5, altura=1.0)
seta(6.45, 9.45, 8.35, 8.75, "Não", (0.10, 0.18))
ax.annotate("", xy=(7.0, 11.0), xytext=(9.0, 8.8),
            arrowprops=dict(arrowstyle="->", lw=1.6, color="#B84949",
                            connectionstyle="arc3,rad=-0.28"))
ax.text(9.45, 10.1, "nova tentativa", fontsize=9.5, color="#9B3636")

# Caminho principal
caixa(5, 8.2, "7. Receber a resposta em JSON", verde, largura=5.0)
seta(5, 9.1, 5, 8.65, "Sim", (0.38, 0))

caixa(5, 6.9, "8. Validar estrutura e quantidade", verde)
caixa(5, 5.6, "9. Preservar o JSON original", verde)
caixa(5, 4.3, "10. Converter JSON em DataFrame", verde)
caixa(5, 3.0, "11. Exibir o DataFrame", verde)
caixa(5, 1.7, "12. Salvar CSV ou Parquet + metadados", verde)
caixa(5, 0.45, "13. Verificar os arquivos na pasta raw", "#CFF3D7")

for y1, y2 in [(7.8, 7.35), (6.5, 6.05), (5.2, 4.75),
               (3.9, 3.45), (2.6, 2.15), (1.3, 0.9)]:
    seta(5, y1, 5, y2)

plt.tight_layout()
plt.show()


## 1. Identificação da equipe

Preencham antes de executar.

- **Equipe:** Grupo 10
- **Integrantes:** Pedro Gabriel Castro da Silva, Ricardo Aguilar Arapa, Lucas Felix Romero, Luiz Eduardo dos Reis, Rodrigo Camargo Vieira.
- **Turma:** Ciência da Computação - Estrutura de Dados II
- **Data da coleta:** 12/09/2026


## 2. Decisões da equipe

As escolhas abaixo são os parâmetros utilizados nesta etapa. A justificativa considera o objetivo da coleta e o limite do recorte, e não somente o tamanho do arquivo.

| Decisão | Escolha da equipe | Justificativa |
|---|---|---|
| ID da medição (`measurement_id`) | `210821089` | Foi escolhida uma medição pública do tipo Ping para obter resultados ICMP já realizados pelo RIPE Atlas, evitando criar uma nova medição nesta etapa. Os metadados serão consultados antes da coleta para confirmar tipo, destino, intervalo, status e sondas. |
| Intervalo consultado | Últimas 1h30 antes da execução (UTC) | Recorte curto e delimitado, suficiente para validar o pipeline de ingestão sem transformar a etapa em uma coleta de longa duração. O intervalo exato será registrado no arquivo de metadados. |
| Duração do intervalo | 1h30 | Mantém o escopo da coleta controlado e permite verificar a granularidade por sonda antes das próximas etapas. |
| Tipo de medição | Ping | É compatível com o objetivo do projeto, pois fornece resultados de RTT e informações de envio/recebimento que poderão ser usadas posteriormente na análise de latência e perda. |
| Formato de saída | CSV | Facilita a inspeção dos registros tabulares, mantendo o JSON original como camada bruta. |

 Antes de escolher, consultem a documentação e verifiquem se a medição é pública, se o ID existe e se o intervalo solicitado possui resultados. Evitem períodos excessivamente longos: eles podem retornar muitos registros ou tornar a consulta lenta.

## 3. Bibliotecas

Serão utilizadas somente as bibliotecas necessárias para esta etapa: `requests` para as requisições HTTP e `pandas` para a camada tabular.

O notebook não precisa instalar `pyarrow`, pois a entrega desta coleta utiliza CSV.


In [ ]:
import json
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import requests

print("Bibliotecas carregadas.")

print("requests:", requests.__version__)
print("pandas:", pd.__version__)


## 4. Montar o Google Drive e preparar a pasta `raw`

O caminho sugerido cria a pasta `ripe_atlas/raw` dentro de `Meu Drive`. A equipe pode alterar somente a pasta do projeto, mantendo uma subpasta chamada `raw`.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PASTA_PROJETO = Path("/content/drive/MyDrive/ripe_atlas")
PASTA_RAW = PASTA_PROJETO / "raw"
PASTA_RAW.mkdir(parents=True, exist_ok=True)

print("Arquivos raw serão salvos em:", PASTA_RAW)


## 5. Definir os parâmetros da consulta

Substituam os valores de exemplo pelas decisões da equipe.

- `MEASUREMENT_IDS`: lista de identificadores numéricos de medições públicas;
- `HORAS_COLETA`: período anterior ao momento da execução (1.5h = 1h30);
- `FORMATO_TABELAR`: use `"csv"` ou `"parquet"`;
- `TIMEOUT_REQUISICAO`: tempo máximo de espera, em segundos.

O período usa timestamps Unix em UTC, formato aceito pela API.


In [ ]:
# Parâmetros definidos pela equipe para consulta histórica
MEASUREMENT_IDS = [
    220245107,  # ID 1
    220244118,  # ID 2
    220243473,  # ID 3
    220243586,  # ID 4
    220243424,  # ID 5
    220243043,  # ID 6
    220242356,  # ID 7
    1014681,    # ID 8
]

FORMATO_TABELAR = "csv"  # opções: "csv" ou "parquet"
TIMEOUT_REQUISICAO = 60
HORAS_COLETA = 1.5

# Asserções para validar os parâmetros de entrada
assert isinstance(MEASUREMENT_IDS, list) and len(MEASUREMENT_IDS) > 0
assert all(isinstance(m_id, int) and m_id > 0 for m_id in MEASUREMENT_IDS)
assert FORMATO_TABELAR in {"csv", "parquet"}
assert TIMEOUT_REQUISICAO > 0

# Definição do intervalo de tempo comum (últimas 1h30 em UTC)
fim_utc = datetime.now(timezone.utc)
inicio_utc = fim_utc - timedelta(hours=HORAS_COLETA)

PARAMETROS_POR_MEDICAO = []

for m_id in MEASUREMENT_IDS:
    url_meta = f"https://atlas.ripe.net/api/v2/measurements/{m_id}/"
    url_results = f"https://atlas.ripe.net/api/v2/measurements/{m_id}/results/"
    
    PARAMETROS_POR_MEDICAO.append({
        "id": m_id,
        "url_metadados": url_meta,
        "url_resultados": url_results,
        "parametros": {
            "start": int(inicio_utc.timestamp()),
            "stop": int(fim_utc.timestamp()),
            "format": "json"
        }
    })

print(f"Total de medições configuradas para coleta: {len(PARAMETROS_POR_MEDICAO)}")

## 6 e 7. Executar Coleta e Tratamento das Medições

A célula abaixo itera sobre todas as medições configuradas, obtendo os metadados, realizando a requisição dos resultados, convertendo os dados para `DataFrame` e salvando a camada `raw` no Google Drive.


In [ ]:
def extrair_rtts(resultados):
    rtts = []
    if isinstance(resultados, list):
        for item in resultados:
            if isinstance(item, dict) and isinstance(item.get("rtt"), (int, float)):
                rtts.append(item["rtt"])
    return rtts

relatorio_coletas = []

for medicao in PARAMETROS_POR_MEDICAO:
    m_id = medicao["id"]
    url_meta = medicao["url_metadados"]
    url_results = medicao["url_resultados"]
    params = medicao["parametros"]
    
    print(f"\n--- Processando Medição ID: {m_id} ---")
    
    # 1. Consultar Metadados
    try:
        response_meta = requests.get(
            url_meta,
            params={"optional_fields": "probes"},
            timeout=TIMEOUT_REQUISICAO,
        )
        response_meta.raise_for_status()
        metadados_medicao = response_meta.json()
    except Exception as erro:
        print(f"Erro ao consultar metadados do ID {m_id}: {erro}")
        continue

    resumo_medicao = {
        "measurement_id": metadados_medicao.get("id"),
        "publica": metadados_medicao.get("is_public"),
        "tipo": metadados_medicao.get("type"),
        "target": metadados_medicao.get("target"),
        "target_ip": metadados_medicao.get("target_ip"),
        "target_asn": metadados_medicao.get("target_asn"),
        "intervalo_segundos": metadados_medicao.get("interval"),
        "status": (metadados_medicao.get("status") or {}).get("name"),
        "is_oneoff": metadados_medicao.get("is_oneoff"),
        "probes_requested": metadados_medicao.get("probes_requested"),
        "probes_scheduled": metadados_medicao.get("probes_scheduled"),
        "participant_count": metadados_medicao.get("participant_count"),
        "start_time": metadados_medicao.get("start_time"),
        "stop_time": metadados_medicao.get("stop_time"),
    }

    if not metadados_medicao.get("is_public") or metadados_medicao.get("type") != "ping":
        print(f"Medição {m_id} desconsiderada: Não é pública ou não é do tipo ping.")
        continue

    # 2. Consultar Resultados
    instante_requisicao_utc = datetime.now(timezone.utc)
    try:
        response = requests.get(
            url_results,
            params=params,
            timeout=TIMEOUT_REQUISICAO,
        )
        response.raise_for_status()
        dados_json = response.json()
    except Exception as erro:
        print(f"Erro ao consultar resultados do ID {m_id}: {erro}")
        continue

    if not isinstance(dados_json, list) or len(dados_json) == 0:
        print(f"Sem registros para a medição {m_id} no intervalo solicitado.")
        continue

    print("Status HTTP Metadados:", response_meta.status_code)
    print("Status HTTP Resultados:", response.status_code)
    print("Quantidade de registros recebidos:", len(dados_json))

    # 3. Transformação em DataFrame
    df_raw = pd.json_normalize(dados_json, sep="_")

    # Integridade e cobertura
    quantidade_sondas = df_raw["prb_id"].nunique(dropna=True) if "prb_id" in df_raw.columns else None
    nulos_totais = int(df_raw.isna().sum().sum())
    
    colunas_hashable = [col for col in df_raw.columns if df_raw[col].apply(lambda x: not isinstance(x, (list, dict))).all()]
    duplicatas = int(df_raw.duplicated(subset=colunas_hashable).sum())

    rcvd_zero = int((df_raw["rcvd"] == 0).sum()) if "rcvd" in df_raw.columns else None
    sent_zero = int((df_raw["sent"] == 0).sum()) if "sent" in df_raw.columns else None

    # Salvamento dos dados na pasta raw
    timestamp_arquivo = instante_requisicao_utc.strftime("%Y%m%dT%H%M%SZ")
    nome_base = f"ripe_atlas_m{m_id}_{timestamp_arquivo}"

    caminho_json = PASTA_RAW / f"{nome_base}.json"
    with caminho_json.open("w", encoding="utf-8") as arquivo:
        json.dump(dados_json, arquivo, ensure_ascii=False, indent=2)

    if FORMATO_TABELAR == "csv":
        caminho_tabela = PASTA_RAW / f"{nome_base}.csv"
        df_raw.to_csv(caminho_tabela, index=False, encoding="utf-8")
    else:
        caminho_tabela = PASTA_RAW / f"{nome_base}.parquet"
        df_raw.to_parquet(caminho_tabela, index=False)

    dicionario_colunas = [
        {"coluna": coluna, "tipo": str(df_raw[coluna].dtype)}
        for coluna in df_raw.columns
    ]

    metadados = {
        "projeto": "Network Failure Predictor",
        "measurement_id": m_id,
        "url_resultados": url_results,
        "url_metadados": url_meta,
        "parametros": params,
        "inicio_utc": inicio_utc.isoformat(),
        "fim_utc": fim_utc.isoformat(),
        "coletado_em_utc": instante_requisicao_utc.isoformat(),
        "status_http_resultados": response.status_code,
        "status_http_metadados": response_meta.status_code,
        "quantidade_registros": len(dados_json),
        "quantidade_colunas_dataframe": len(df_raw.columns),
        "quantidade_sondas_distintas": quantidade_sondas,
        "registros_com_rcvd_zero": rcvd_zero,
        "registros_com_sent_zero": sent_zero,
        "registros_duplicados": duplicatas,
        "metadados_medicao": resumo_medicao,
        "colunas": dicionario_colunas,
        "arquivo_json": caminho_json.name,
        "arquivo_tabelar": caminho_tabela.name,
        "arquivo_metadados": f"{nome_base}_metadata.json",
        "observacao": (
            "Camada raw: o JSON original é preservado e o CSV é uma representação tabular. "
            "Não foram realizadas limpeza, agregação, rotulação ou treinamento nesta etapa."
        ),
    }

    caminho_metadados = PASTA_RAW / f"{nome_base}_metadata.json"
    with caminho_metadados.open("w", encoding="utf-8") as arquivo:
        json.dump(metadados, arquivo, ensure_ascii=False, indent=2)

    relatorio_coletas.append({
        "id": m_id,
        "registros": len(dados_json),
        "json": caminho_json.name,
        "tabela": caminho_tabela.name,
        "metadados": caminho_metadados.name
    })

print("\nProcessamento finalizado!")

## 8. Verificar se a entrega foi salva

A célula abaixo exibe o resumo dos arquivos gerados para todas as medições processadas.


In [ ]:
df_relatorio = pd.DataFrame(relatorio_coletas)
display(df_relatorio)
print("Coleta concluída e salva na pasta raw.")


### 9. Contribuições Individuais

Integrante 1 — `Pedro Gabriel Castro da Silva`

* **Atividade realizada:**
  `participação na criação e preparação da base inicial do projeto para a coleta de dados.`

* **Parte do trabalho relacionada:**
  `montagem da estrutura e coleta dos dados`

* **Tempo dedicado (aproximado):**
  `1h30`

* **Evidência da contribuição:**
  `https://drive.google.com/drive/folders/1IdQxeehfBrU3i9RdZ0v8qTYGiZeDMAY-?hl=pt-br`

* **Explicação da evidência:**
  `commit realizado no repositório contendo a estrutura inicial do projeto e os arquivos-base utilizados para a coleta de dados.`

Integrante 2 — `Lucas Felix Romero`

* **Atividade realizada:**
  `atualização da estrutura inicial do projeto e criação/organização do repositório de evidências da coleta`

* **Parte do trabalho relacionada:**
  `estrutura da coleta de dados e organização do repositório de evidências`

* **Tempo dedicado (aproximado):**
  `2h`

* **Evidência da contribuição:**
  `https://drive.google.com/drive/folders/1IdQxeehfBrU3i9RdZ0v8qTYGiZeDMAY-?hl=pt-br`

* **Explicação da evidência:**
  `a pasta reúne os materiais de evidência utilizados nesta etapa. A contribuição está relacionada à organização da estrutura inicial e ao registro dos arquivos utilizados para documentar a coleta.`

Integrante 3 — `Ricardo Aguilar Arapa`

* **Atividade realizada:**
  `Não participou ativamente desta etapa`

* **Parte do trabalho relacionada:**
  `Não se aplica`

* **Tempo dedicado (aproximado):**
  `Não se aplica`

* **Evidência da contribuição:**
  `Não se aplica pois nao houve contribuição direta nesta etapa`

* **Explicação da evidência:**
  `Não se aplica`


Integrante 4 — `Luiz Eduardo dos Reis`

* **Atividade realizada:**
  `Não participou ativamente desta etapa`

* **Parte do trabalho relacionada:**
  `Não se aplica`

* **Tempo dedicado (aproximado):**
  `Não se aplica`

* **Evidência da contribuição:**
  `Não se aplica pois não houve contribuição direta nesta etapa`

* **Explicação da evidência:**
  `Não se aplica`

Integrante 5 — `Rodrigo Camargo Vieira`

* **Atividade realizada:**
  `Pesquisa de measurements, avaliação das alterações`

* **Parte do trabalho relacionada:**
  `Estrutura da coleta de dados e atualizações da measurements`

* **Tempo dedicado (aproximado):**
  `1h`

* **Evidência da contribuição:**
  `https://drive.google.com/drive/folders/1IdQxeehfBrU3i9RdZ0v8qTYGiZeDMAY-?hl=pt-br`

* **Explicação da evidência:**
  `A pasta mostra evidências relacionadas a esta etapa do projeto`







**Observação**:

Evidências aceitas
1. histórico de edição de documento compartilhado;
2. commit ou pull request no GitHub;
3. arquivo ou trecho de código produzido;
4. relatório, planilha, diagrama, apresentação ou rascunho;
5. registro de tarefa no Trello, GitHub Projects ou ferramenta semelhante;
6. registro de testes realizados;
7. print de reunião, conversa ou e-mail relacionado à atividade;
8. outro material que demonstre claramente a contribuição individual.

Sendo que a evidência deverá identificar o aluno e estar relacionada à atividade declarada. Sempre que possível, apresente materiais com data, autoria ou histórico de edição. Verifique se todos os links estão acessíveis. Trabalhos realizados em conjunto também devem indicar a contribuição específica de cada integrante. Respostas genéricas, como “ajudei no trabalho”, “fiz a pesquisa” ou “participei do código”, não serão consideradas suficientes. Prints de conversas podem complementar a comprovação, mas não devem ser a única evidência quando houver um produto verificável, como código, documento, planilha ou apresentação.
A ausência de descrição ou de evidência poderá impedir a validação da contribuição individual.

## 10. Checklist da equipe

- [x] Identificamos os integrantes e a turma;
- [x] Justificamos o ID da medição e o intervalo escolhido no notebook;
- [x] A requisição de resultados terminou com status HTTP 200;
- [x] A consulta de metadados terminou com status HTTP 200;
- [x] A medição foi conferida como pública e do tipo Ping;
- [x] O JSON retornou pelo menos um registro;
- [x] Exibimos o `DataFrame` no notebook;
- [x] Verificamos a quantidade de sondas distintas e a distribuição de registros por `prb_id`;
- [x] Realizamos validações básicas de nulos, `sent`, `rcvd` e duplicidades;
- [x] Salvamos o JSON original na pasta `raw`;
- [x] Salvamos o DataFrame em CSV na pasta `raw`;
- [x] Mantivemos o arquivo de metadados da coleta;
- [x] Registramos o dicionário das colunas;
- [x] Não realizamos limpeza, agregação, rotulação ou treinamento nesta etapa.

> Os itens acima serão confirmados pelas saídas da execução. Se uma execução futura falhar em algum item, a caixa correspondente deve ser desmarcada até que a evidência seja recuperada.

### Registro final da equipe

Foi realizada uma coleta de resultados de medições públicas do tipo Ping na API RIPE Atlas, iterando sobre a lista de IDs definidos. Antes da consulta dos resultados, foram obtidos os metadados de cada medição para confirmar sua existência, tipo, publicidade, destino, intervalo, status e quantidade de sondas. Foi definido um recorte de 1h30, com início e fim registrados em UTC, e o formato tabular escolhido foi CSV.

Os resultados foram mantidos na camada `raw`: os arquivos JSON originais foram preservados sem transformação, os conteúdos foram convertidos para `DataFrame` e salvos em CSV, e arquivos de metadados registram os parâmetros, horários, status HTTP, quantidade de registros, cobertura por sondas e estrutura das colunas. Nesta etapa não foram realizadas limpeza, agregação, rotulação ou treinamento.
